In [30]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split , GridSearchCV , KFold , cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score , mean_squared_error , mean_absolute_error
from sklearn.compose import ColumnTransformer , TransformedTargetRegressor
from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler , OneHotEncoder , OrdinalEncoder , PowerTransformer
from sklearn.base import BaseEstimator , TransformerMixin
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice

In [2]:
dagshub.init(repo_owner="mridul0010" , repo_name="food-delivery-time-prediction" , mlflow=True)

Accessing as mridul0010

Initialized MLflow to track repo "mridul0010/food-delivery-time-prediction"

Repository mridul0010/food-delivery-time-prediction initialized!

In [3]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow")

In [4]:
mlflow.set_experiment("2. Hyperparameter Tuning - XGB")

2026/07/05 16:14:35 INFO mlflow.tracking.fluent: Experiment with name '2. Hyperparameter Tuning - XGB' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/fdfb360dd9ef4f79a3f3af549e6f3a3f', creation_time=1783248276959, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1783248276959, lifecycle_stage='active', name='2. Hyperparameter Tuning - XGB', tags={}, trace_location=None, workspace='default'>

In [5]:
df = pd.read_csv('../data/Cleaned Delivery Dataset.csv')

In [6]:
pd.set_option('display.max_columns' , None)

In [7]:
df.drop(columns=['Delivery_Agent'] , inplace=True)

In [8]:
df['Order_Datetime'] = pd.to_datetime(df['Order_Datetime'])
df['Pickup_Datetime'] = pd.to_datetime(df['Pickup_Datetime'])

In [9]:
class FeatureEngineering(BaseEstimator, TransformerMixin):
    
    def __init__(self):
        self.rating_bins = None

    def fit(self, X, y=None):
        X = X.copy()
        
        # Store bins for consistent transformation
        self.rating_bins = pd.qcut(
            X['Delivery_person_Ratings'],
            q=3,
            retbins=True,
            duplicates='drop'
        )[1]
        
        return self

    def transform(self, X):
        X = X.copy()

        # Datetime conversion
        X['Order_Datetime'] = pd.to_datetime(X['Order_Datetime'], errors='coerce')
        X['Pickup_Datetime'] = pd.to_datetime(X['Pickup_Datetime'], errors='coerce')

        # Rating group
        X["delivery_rating_group"] = pd.cut(
            X['Delivery_person_Ratings'],
            bins=self.rating_bins,
            labels=['Low', 'Medium', 'High'],
            include_lowest=True
        )

        # Age group
        X["age_group"] = pd.cut(
            X['Delivery_person_Age'],
            bins=[14, 25, 35, 60],
            labels=['Young', 'Adult', 'Senior']
        )

        # Distance group
        X["distance_group"] = pd.cut(
            X['distance_km'],
            bins=[0, 5, 10, 25],
            labels=['Short Distance', 'Medium Distance', 'Long Distance']
        )

        # Time features
        X['Prep_Time(min)'] = (
            X['Pickup_Datetime'] - X['Order_Datetime']
        ).dt.total_seconds() / 60

        X['Order_hour'] = X['Order_Datetime'].dt.hour
        X['Order_day'] = X['Order_Datetime'].dt.day_name()

        X['isWeekend'] = X['Order_day'].isin(["Saturday", "Sunday"]).astype(int)

        X['Time_Of_Day'] = pd.cut(
            X['Order_hour'],
            bins=[0, 6, 12, 18, 24],
            labels=["Night", "Morning", "Afternoon", "Evening"],
            include_lowest=True
        )

        # Drop raw datetime
        X = X.drop(columns=['Order_Datetime', 'Pickup_Datetime'], errors='ignore')

        return X

    def get_feature_names_out(self , input_features = None):
        return input_features

In [10]:
X = df.drop(columns=['Time_taken (min)'])
y = df['Time_taken (min)']

In [11]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42)

In [12]:
X

,City,Zone,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,distance_km,Order_Datetime,Pickup_Datetime,Weather_conditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City_Type
0,Dehradun,Zone17,36,4.2,30.327968,78.046106,30.397968,78.116106,10.28,2022-12-02 21:55:00,2022-12-02 22:10:00,Fog,Jam,Good,Snack,motorcycle,3,No,Metropolitian
1,Kochi,Zone16,21,4.7,10.003064,76.307589,10.043064,76.347589,6.24,2022-02-13 14:55:00,2022-02-13 15:05:00,Stormy,High,Average,Meal,motorcycle,1,No,Metropolitian
2,Pune,Zone13,23,4.7,18.562450,73.916619,18.652450,74.006619,13.79,2022-04-03 17:30:00,2022-04-03 17:40:00,Sandstorms,Medium,Average,Drinks,scooter,1,No,Metropolitian
3,Ludhiana,Zone15,34,4.3,30.899584,75.809346,30.919584,75.829346,2.93,2022-02-13 09:20:00,2022-02-13 09:30:00,Sandstorms,Low,poor,Buffet,motorcycle,0,No,Metropolitian
4,Kanpur,Zone14,24,4.7,26.463504,80.372929,26.593504,80.502929,19.40,2022-02-14 19:50:00,2022-02-14 20:05:00,Fog,Jam,Average,Snack,scooter,1,No,Metropolitian
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39637,Bangalore,Zone16,28,4.9,13.029198,77.570997,13.059198,77.600997,4.66,2022-03-30 21:55:00,2022-03-30 22:05:00,Sandstorms,Jam,Average,Meal,scooter,1,No,Metropolitian
39638,Ranchi,Zone16,35,4.2,23.371292,85.327872,23.481292,85.437872,16.60,2022-08-03 21:45:00,2022-08-03 21:55:00,Windy,Jam,Good,Drinks,motorcycle,1,No,Metropolitian
39639,Chennai,Zone08,30,4.9,13.022394,80.242439,13.052394,80.272439,4.66,2022-11-03 23:50:00,2022-11-04 00:05:00,Cloudy,Low,Average,Drinks,scooter,0,No,Metropolitian
39640,Coimbatore,Zone11,20,4.7,11.001753,76.986241,11.041753,77.026241,6.23,2022-07-03 13:35:00,2022-07-03 13:40:00,Cloudy,High,poor,Snack,motorcycle,1,No,Metropolitian


In [13]:
numerical = ['Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude',
       'Restaurant_longitude', 'Delivery_location_latitude',
       'Delivery_location_longitude', 
       'multiple_deliveries', 'distance_km', 'Prep_Time(min)', 'Order_hour',
       'isWeekend']

ohe_col = ['City', 'Zone', 'Weather_conditions',
       'Type_of_order', 'Type_of_vehicle',
       'City_Type','Order_day', 'Time_Of_Day']

ordinal_col = ['Road_traffic_density' , 'Vehicle_condition' , 'Festival',
       'delivery_rating_group' , 'age_group' , 'distance_group']

In [14]:
Road_traffic_density = ['Low', 'Medium', 'High', 'Jam']
Vehicle_condition = ['poor', 'Average', 'Good' , 'Excellent']
Festival = ['No', 'Yes']
delivery_rating_group = ['Low', 'Medium', 'High']
age_group = ['Young', 'Adult', 'Senior']         
distance_group = ['Short Distance', 'Medium Distance', 'Long Distance']

In [15]:
transformer = ColumnTransformer(
    transformers=[
        ("ohe" , OneHotEncoder(sparse_output=False , handle_unknown='ignore') , ohe_col),
        ('oe' , OrdinalEncoder(categories=[
            Road_traffic_density , Vehicle_condition,
            Festival , delivery_rating_group,
            age_group , distance_group
        ]) , ordinal_col),
        ('Scaling' , StandardScaler() , numerical)
    ],remainder='passthrough'
)

In [16]:
pipeline = Pipeline(
    [
        ("FeatureEngineering" , FeatureEngineering()),
        ("ColumnTransformer" , transformer)
    ]
) 

In [17]:
X_train_trans = pipeline.fit_transform(X_train)
X_test_trans = pipeline.transform(X_test)

In [18]:
pt = PowerTransformer()

In [19]:
def objective(trial):
    # Safe sequential execution or thread-isolated nested runs
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 1, 40),
            "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2),
            "subsample": trial.suggest_float("subsample", 0.4, 1),
            "min_child_weight": trial.suggest_int("min_child_weight", 5, 20),
            "gamma": trial.suggest_float("gamma", 0, 10), 
            "reg_lambda": trial.suggest_float("reg_lambda", 0, 100),
            "random_state": 42,
            "n_jobs": -1,
        }

        mlflow.log_params(params)

        xgb = XGBRegressor(**params)
        model = TransformedTargetRegressor(regressor=xgb, transformer=pt)

        # REMOVED: model.fit(...) <- cross_val_score handles this internally
        cv_score = cross_val_score(
            model,
            X_train_trans, 
            y_train, 
            cv=10, 
            scoring="neg_mean_absolute_error",
            n_jobs=-1
        )

        mean_score = -(cv_score.mean())
        mlflow.log_metric("cross_val_error", mean_score)

        return mean_score

In [20]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # Fixed n_jobs=1 to avoid MLflow concurrent logging crashes
    study.optimize(objective, n_trials=50, n_jobs=1, show_progress_bar=True)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_score", study.best_value)

    # Re-train using the best parameters
    best_xgb = XGBRegressor(**study.best_params)
    best_model = TransformedTargetRegressor(regressor=best_xgb, transformer=pt)
    best_model.fit(X_train_trans, y_train)

    # Evaluate
    y_pred_train = best_model.predict(X_train_trans)
    y_pred_test = best_model.predict(X_test_trans)

    scores = cross_val_score(
        best_model,
        X_train_trans,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5, 
        n_jobs=-1
    )

    mlflow.log_metric("Training_error", mean_absolute_error(y_train, y_pred_train))
    mlflow.log_metric("Test_error", mean_absolute_error(y_test, y_pred_test))
    mlflow.log_metric("Training_r2", r2_score(y_train, y_pred_train))
    mlflow.log_metric("Test_r2", r2_score(y_test, y_pred_test))
    mlflow.log_metric("cross_val", -scores.mean())

    # Generate and log plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    # Loging the best model to MLflow
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="model_XGB",  
        serialization_format="cloudpickle"
    )

[I 2026-07-05 16:14:36,573] A new study created in memory with name: no-name-6aa9f7fe-6a0b-4c2a-b13f-df6cce062cfc
  0%|          | 0/50 [00:00<?, ?it/s]

🏃 View run selective-whale-658 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2/runs/0efbb35f8d4c4983bec637d7dd99350b
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2


Best trial: 0. Best value: 3.02804:   2%|▏         | 1/50 [00:06<04:58,  6.10s/it]

[I 2026-07-05 16:14:43,267] Trial 0 finished with value: 3.028035378456116 and parameters: {'n_estimators': 105, 'max_depth': 27, 'learning_rate': 0.1737786174856056, 'subsample': 0.835727649064288, 'min_child_weight': 20, 'gamma': 1.3397138495437244, 'reg_lambda': 40.19895743413683}. Best is trial 0 with value: 3.028035378456116.
🏃 View run upbeat-slug-82 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2/runs/10634e1459854dbd9b16b1097c4bdd7c
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2


Best trial: 0. Best value: 3.02804:   4%|▍         | 2/50 [00:11<04:32,  5.68s/it]

...

[I 2026-07-05 16:21:13,917] Trial 48 finished with value: 3.095129466056824 and parameters: {'n_estimators': 130, 'max_depth': 19, 'learning_rate': 0.07504041743161306, 'subsample': 0.8644769102446638, 'min_child_weight': 17, 'gamma': 5.99432172703587, 'reg_lambda': 22.068607892937237}. Best is trial 46 with value: 2.9965721368789673.
🏃 View run zealous-chimp-736 at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2/runs/dd87c1b590e44fcebdc74dd6b5998a9c
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2


Best trial: 46. Best value: 2.99657: 100%|██████████| 50/50 [06:50<00:00,  8.21s/it]


[I 2026-07-05 16:21:27,754] Trial 49 finished with value: 3.002767777442932 and parameters: {'n_estimators': 182, 'max_depth': 13, 'learning_rate': 0.03691405355029673, 'subsample': 0.7777266401692674, 'min_child_weight': 19, 'gamma': 0.4030595669703477, 'reg_lambda': 9.381882472789783}. Best is trial 46 with value: 2.9965721368789673.


2026/07/05 16:21:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2/runs/e84df539acd742d599968f45f28bb4dc
🧪 View experiment at: https://dagshub.com/mridul0010/food-delivery-time-prediction.mlflow/#/experiments/2


In [21]:
study.best_value

2.9965721368789673

In [22]:
study.best_params

{'n_estimators': 135,
 'max_depth': 14,
 'learning_rate': 0.08141804248814871,
 'subsample': 0.897256521400417,
 'min_child_weight': 16,
 'gamma': 0.7199050429299239,
 'reg_lambda': 14.533893839965948}

In [23]:
best_xgb = XGBRegressor(**study.best_params)

model = TransformedTargetRegressor(regressor=best_xgb , transformer=pt)

model.fit(X_train_trans , y_train)

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.","XGBRegressor(...ree=None, ...)"
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",PowerTransformer()
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",None
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,87
regressor_ regressor_: objectFitted regressor.,XGBRegressor,"XGBRegressor(...ree=None, ...)"
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,PowerTransformer,PowerTransformer()
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None


In [24]:
y_pred = model.predict(X_test_trans)

r2_score(y_test , y_pred)

0.8364468216896057

In [25]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [26]:
fig_history.show()

In [27]:
fig_parallel 

In [28]:
fig_importance

In [29]:
fig_slice